# Model Training

## Import necessary libraries

In [8]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [10]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="DataPreparingSpark")

## Loading Datasets

In [11]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="raw_data_path")

df = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

df.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|  ticket_id|               type|        organization|             comment|               photo|         photo_after|            coords|             address|subdistrict|district|     province|           timestamp|    state|star|count_reopen|       last_activity|
+-----------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+-----------+--------+-------------+--------------------+---------+----+------------+--------------------+
|2021-FYJTFP|        {ความสะอาด}|          เขตบางซื่อ|             ขยะเยอะ|https://storage.g...|                NULL|100.53084,13.81865|12/14 ถนน กรุงเทพ...|       NULL|    NULL|กรุงเทพมหานคร|2021-09-03 19:51:..

---

## Applying Cleansing Pipeline

In [12]:
from src.pipelines_spark import CleansingPipelineSpark

cleansing_pipeline = CleansingPipelineSpark(spark, sedona)
df_cleansed = cleansing_pipeline.transform(df)

df_cleansed.show(10)

+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|  ticket_id|               type|        organization|             comment|             address|subdistrict|district|     province|timestamp_date|timestamp_month|timestamp_year|last_activity_date|last_activity_month|last_activity_year|resolution_time|longitude|latitude|status|
+-----------+-------------------+--------------------+--------------------+--------------------+-----------+--------+-------------+--------------+---------------+--------------+------------------+-------------------+------------------+---------------+---------+--------+------+
|2021-CGPMUN|{น้ำท่วม,ร้องเรียน}|เขตประเวศ,ฝ่ายโยธ...|น้ำท่วมเวลาฝนตกแล...|189 เฉลิมพระเกียร...|    หนองบอน|  ประเวศ|กรุงเทพมหานคร|            19|              9|    

In [13]:
df_cleansed.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp_date: integer (nullable = true)
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- last_activity_date: integer (nullable = true)
 |-- last_activity_month: integer (nullable = true)
 |-- last_activity_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- status: string (nullable = true)



---

## Testing Transformers

### Data Filter Transformer

In [14]:
from src.pipelines_spark import DataFilterTransformerSpark

dft = DataFilterTransformerSpark()
df_transformed = dft.transform(df_cleansed)

df_transformed.printSchema()

root
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)



### Address Encoder

In [15]:
from src.pipelines_spark import AddressEncoderSpark

ae = AddressEncoderSpark()
df_encoded = ae.transform(df_cleansed)

df_cleansed.select(["district", "subdistrict", "latitude", "longitude"]).show(
    10, truncate=False
)
df_encoded.select(["address_encoded", "latlong_encoded"]).show(10, truncate=False)

+--------+-----------+--------+---------+
|district|subdistrict|latitude|longitude|
+--------+-----------+--------+---------+
|ประเวศ  |หนองบอน    |13.67891|100.66709|
|สาทร    |ยานนาวา    |13.7206 |100.52649|
|ลาดพร้าว|ลาดพร้าว   |13.8228 |100.59165|
|ลาดพร้าว|ลาดพร้าว   |13.8091 |100.59131|
|ดุสิต   |ดุสิต      |13.77832|100.50848|
|ประเวศ  |หนองบอน    |13.67083|100.6469 |
|ประเวศ  |ประเวศ     |13.72812|100.65617|
|ประเวศ  |หนองบอน    |13.68158|100.6544 |
|ประเวศ  |หนองบอน    |13.68735|100.64844|
|ประเวศ  |ประเวศ     |13.71887|100.68837|
+--------+-----------+--------+---------+
only showing top 10 rows

+----------------------------+--------------------+
|address_encoded             |latlong_encoded     |
+----------------------------+--------------------+
|(2048,[834,1804],[1.0,1.0]) |[13.67891,100.66709]|
|(2048,[348,426],[1.0,1.0])  |[13.7206,100.52649] |
|(2048,[802,1656],[1.0,1.0]) |[13.8228,100.59165] |
|(2048,[802,1656],[1.0,1.0]) |[13.8091,100.59131] |
|(2048,[1025,1114],[1.

### Organization Encoder

In [16]:
from src.pipelines_spark import OrganizationEncoderSpark

oe = OrganizationEncoderSpark()
df_encoded = oe.transform(df_cleansed)

df_cleansed.select(["organization"]).show(10, truncate=False)
df_encoded.select(["organization_encoded"]).show(10, truncate=False)

+------------------------------------------------------------+
|organization                                                |
+------------------------------------------------------------+
|เขตประเวศ,ฝ่ายโยธา เขตประเวศ                                |
|เขตสาทร                                                     |
|เขตลาดพร้าว,ฝ่ายโยธา เขตลาดพร้าว                            |
|เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์                    |
|เขตดุสิต                                                    |
|เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ,ฝ่ายรักษาความสะอาดฯ เขตประเวศ|
|เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ                              |
|เขตประเวศ,ฝ่ายโยธา เขตประเวศ                                |
|เขตประเวศ,ฝ่ายโยธา เขตประเวศ                                |
|เขตประเวศ,สำนักสิ่งแวดล้อม กทม.,ฝ่ายสิ่งแวดล้อมฯ เขตประเวศ  |
+------------------------------------------------------------+
only showing top 10 rows

+-------------------------------+
|organization_encoded           |
+-----------------------

### Type Encoder

In [17]:
from src.pipelines_spark import TypeEncoderSpark

te = TypeEncoderSpark()
df_encoded = te.transform(df_cleansed)

df_cleansed.select(["type"]).show(10, truncate=False)
df_encoded.select(["type_encoded"]).show(10, truncate=False)

+-------------------+
|type               |
+-------------------+
|{น้ำท่วม,ร้องเรียน}|
|{สะพาน}            |
|{น้ำท่วม,ถนน}      |
|{}                 |
|{}                 |
|{ความสะอาด}        |
|{}                 |
|{ท่อระบายน้ำ}      |
|{}                 |
|{ความสะอาด}        |
+-------------------+
only showing top 10 rows

+--------------------+
|type_encoded        |
+--------------------+
|(25,[7,8],[1.0,1.0])|
|(25,[14],[1.0])     |
|(25,[0,8],[1.0,1.0])|
|(25,[1],[1.0])      |
|(25,[1],[1.0])      |
|(25,[3],[1.0])      |
|(25,[1],[1.0])      |
|(25,[9],[1.0])      |
|(25,[1],[1.0])      |
|(25,[3],[1.0])      |
+--------------------+
only showing top 10 rows



### Encoder Pipeline

In [18]:
from src.pipelines_spark import EncoderPipelineSpark

ep = EncoderPipelineSpark()
df_encoded = ep.transform(df_cleansed)

df_cleansed.select(
    ["district", "subdistrict", "latitude", "longitude", "organization", "type"]
).show(10, truncate=False)
df_encoded.select(
    ["address_encoded", "latlong_encoded", "organization_encoded", "type_encoded"]
).show(10, truncate=False)

+--------+-----------+--------+---------+------------------------------------------------------------+-------------------+
|district|subdistrict|latitude|longitude|organization                                                |type               |
+--------+-----------+--------+---------+------------------------------------------------------------+-------------------+
|ประเวศ  |หนองบอน    |13.67891|100.66709|เขตประเวศ,ฝ่ายโยธา เขตประเวศ                                |{น้ำท่วม,ร้องเรียน}|
|สาทร    |ยานนาวา    |13.7206 |100.52649|เขตสาทร                                                     |{สะพาน}            |
|ลาดพร้าว|ลาดพร้าว   |13.8228 |100.59165|เขตลาดพร้าว,ฝ่ายโยธา เขตลาดพร้าว                            |{น้ำท่วม,ถนน}      |
|ลาดพร้าว|ลาดพร้าว   |13.8091 |100.59131|เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์                    |{}                 |
|ดุสิต   |ดุสิต      |13.77832|100.50848|เขตดุสิต                                                    |{}                 |
|ประเวศ  |หนองบอ

---

## Applying Model Preparation Pipeline

In [19]:
from src.pipelines_spark import ModelPrepPipelineSpark

preparing_pipeline = ModelPrepPipelineSpark()
df_prepared = preparing_pipeline.transform(df_cleansed)

df_prepared.show(10)

+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|     address_encoded|     latlong_encoded|organization_encoded|        type_encoded|
+---------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+
|              9|          2021|            275|(2048,[834,1804],...|[13.67891,100.66709]|(2058,[9,59],[1.0...|(25,[7,8],[1.0,1.0])|
|              9|          2021|            253|(2048,[348,426],[...| [13.7206,100.52649]|   (2058,[55],[1.0])|     (25,[14],[1.0])|
|             12|          2021|            246|(2048,[802,1656],...| [13.8228,100.59165]|(2058,[35,112],[1...|(25,[0,8],[1.0,1.0])|
|             12|          2021|            456|(2048,[802,1656],...| [13.8091,100.59131]|(2058,[35,169],[1...|      (25,[1],[1.0])|
|             12|          2021|            516|(2048,[1025,1114]...|

In [20]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Saving Files

In [21]:
df_pandas = df_prepared.toPandas()
df_pandas.head()

,timestamp_month,timestamp_year,resolution_time,address_encoded,latlong_encoded,organization_encoded,type_encoded
0,9,2021,275,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[13.67891, 100.66709]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, ..."
1,9,2021,253,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[13.7206, 100.52649]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,12,2021,246,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[13.8228, 100.59165]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
3,12,2021,456,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[13.8091, 100.59131]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,12,2021,516,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[13.77832, 100.50848]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [22]:
import pandas as pd
from src.utils import get_data_dir

save_name = "model_training_spark.csv"
save_path = get_data_dir() / "processed" / save_name

pd.DataFrame.to_csv(
    df_pandas,
    save_path,
    index=False,
)

In [23]:
spark.stop()

---